# 04 — Modélisation & Validation Croisée

Ce notebook entraîne et évalue plusieurs modèles de classification sur le jeu de données openFDA FAERS afin de prédire **seriousnesshospitalization** (binaire : 0 = pas d'hospitalisation, 1 = hospitalisation). La classe cible est déséquilibrée (~15–25% de positifs), nous comparons donc trois stratégies de rééchantillonnage sur quatre classifieurs à l'aide d'une validation croisée stratifiée k-fold. Le jeu de test n'est **jamais utilisé** ici.

## Section 1 — Imports

Nous importons toutes les bibliothèques nécessaires dès le début. `imblearn.pipeline.Pipeline` (aliasée en `ImbPipeline`) est utilisée **exclusivement** pour tous les pipelines afin que les étapes de rééchantillonnage soient correctement appliquées uniquement aux folds d'entraînement lors de la validation croisée — l'utilisation du Pipeline natif de sklearn ne gérerait pas correctement SMOTE ou RandomUnderSampler.

In [1]:
import pandas as pd
import numpy as np
import joblib

# sklearn models & CV utilities
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# XGBoost
from xgboost import XGBClassifier

# imbalanced-learn resampling + pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Global random state
RANDOM_STATE = 42

print('All imports successful.')

All imports successful.


## Section 2 — Chargement des Données d'Entraînement

Nous chargeons uniquement `train.csv` (découpage stratifié à 70% préparé par le Membre 3). La colonne cible `seriousnesshospitalization` est séparée des variables explicatives. Nous calculons également `scale_pos_weight` — le ratio des négatifs sur les positifs — que XGBoost utilise en interne pour gérer le déséquilibre des classes sans rééchantillonnage explicite.

In [2]:
TARGET = 'seriousnesshospitalization'

train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

print(f'Training set shape  : {X_train.shape}')
print(f'Target distribution :\n{y_train.value_counts(normalize=True).round(4)}')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'\nscale_pos_weight = {scale_pos_weight:.4f}')

Training set shape  : (5422, 14)
Target distribution :
seriousnesshospitalization
0    0.7615
1    0.2385
Name: proportion, dtype: float64

scale_pos_weight = 3.1933


## Section 3 — Chargement du Préprocesseur Ajusté

Le préprocesseur `ColumnTransformer` a été construit et **ajusté sur X_train** par le Membre 3 (combinant KNNImputer + RobustScaler pour les variables numériques, OrdinalEncoder pour `worst_reaction_outcome`, et OneHotEncoder pour les variables nominales). Nous le chargeons tel quel et **ne le réajustons pas**, afin d'éviter toute fuite de données et de respecter le pipeline préexistant.

In [3]:
from sklearn.base import clone

preprocessor_template = joblib.load('../models/preprocessor.joblib')
preprocessor = clone(preprocessor_template) 

print('Preprocessor loaded successfully.')
print(f'Type : {type(preprocessor)}')

Preprocessor loaded successfully.
Type : <class 'sklearn.compose._column_transformer.ColumnTransformer'>


## Section 4 — Définition des Modèles de Référence

### Choix des 4 modèles

Le protocole Phase 3 exige au minimum 4 modèles avec une **diversité algorithmique**.
Le professeur propose 6 candidats : Régression Logistique, Arbre de Décision, Random Forest, XGBoost/LightGBM, SVM, et MLP.

Nous retenons les 4 suivants :

**Régression Logistique** — retenue comme baseline. Rapide, interprétable, et solide sur des données tabulaires linéairement séparables. Elle établit un plancher de performance auquel tous les autres modèles seront comparés.

**Arbre de Décision** — retenu pour sa capacité à capturer des non-linéarités et pour son interprétabilité maximale. Il représente la famille des modèles simples à base d'arbres avant agrégation.

**Random Forest** — retenu comme représentant des méthodes d'ensemble par bagging. Il corrige l'instabilité de l'arbre simple en agrégeant 100 arbres indépendants, réduisant ainsi la variance sans augmenter le biais.

**XGBoost** — retenu comme représentant des méthodes d'ensemble par boosting. Contrairement au Random Forest, les arbres sont construits séquentiellement : 
chaque arbre corrige les erreurs du précédent. C'est historiquement le meilleur algorithme sur des données tabulaires mixtes, et il intègre nativement `scale_pos_weight` pour gérer le déséquilibre de classe.

### Pourquoi SVM et MLP sont exclus

**SVM écarté** : le professeur lui-même note qu'il est *"très lent sur > 10 000 lignes"*. Notre dataset post-nettoyage contient ~7 700 lignes avec un espace de features élevé après OneHotEncoding — le SVM serait impraticable dans notre contrainte 
de temps de calcul.

**MLP écarté** : le MLP *"nécessite plus de données et de tuning"* (professeur). Notre dataset de ~7 700 lignes est sur la limite basse pour qu'un réseau de neurones généralise correctement. De plus, nos features sont tabulaires et mixtes (numériques + catégorielles encodées), un profil où les méthodes à base d'arbres dominent systématiquement le MLP sans tuning extensif.

Quatre classifieurs sont instanciés avec des hyperparamètres par défaut. `class_weight='balanced'` est défini pour les modèles sklearn afin de pénaliser proportionnellement la mauvaise classification de la classe minoritaire. Pour XGBoost, nous utilisons le `scale_pos_weight` calculé ci-dessus, ce qui atteint le même effet nativement. Tous les modèles utilisent `random_state=42` pour la reproductibilité.

In [22]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=RANDOM_STATE
    ),
}

print('Models defined:')
for name in models:
    print(f'  - {name}')

Models defined:
  - Logistic Regression
  - Decision Tree
  - Random Forest
  - XGBoost


## Section 5 — Boucle de Validation Croisée (4 Modèles × 3 Stratégies)

Nous évaluons 12 configurations (4 modèles × 3 stratégies de rééchantillonnage) à l'aide d'une **validation croisée stratifiée à 5 folds** scorée sur le **F1 macro**. Tous les pipelines utilisent `ImbPipeline` afin que SMOTE et RandomUnderSampler soient appliqués uniquement aux folds d'entraînement — jamais au fold de validation — évitant ainsi toute fuite de données. Pour les stratégies SMOTE et Undersampling, le `scale_pos_weight` de XGBoost est réinitialisé à 1 car le rééchantillonnage équilibre déjà la distribution des classes.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for model_name, model in models.items():

    # ------------------------------------------------------------------ #
    # Strategy 1 — No resampling                                         #
    # ------------------------------------------------------------------ #
    strategy_name = 'No Resampling'
    print(f'Training {model_name} — {strategy_name}...')

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')
    results.append({
        'Model': model_name,
        'Strategy': strategy_name,
        'Mean F1': scores.mean(),
        'Std F1': scores.std()
    })

    # ------------------------------------------------------------------ #
    # Strategy 2 — SMOTE oversampling                                    #
    # ------------------------------------------------------------------ #
    strategy_name = 'SMOTE'
    print(f'Training {model_name} — {strategy_name}...')

    # For SMOTE, data is balanced → reset scale_pos_weight to 1 for XGBoost
    if model_name == 'XGBoost':
        smote_model = XGBClassifier(
            scale_pos_weight=1,
            eval_metric='logloss',
            random_state=RANDOM_STATE
        )
    else:
        smote_model = model

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('classifier', smote_model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')
    results.append({
        'Model': model_name,
        'Strategy': strategy_name,
        'Mean F1': scores.mean(),
        'Std F1': scores.std()
    })

    # ------------------------------------------------------------------ #
    # Strategy 3 — Random Undersampling                                  #
    # ------------------------------------------------------------------ #
    strategy_name = 'Undersampling'
    print(f'Training {model_name} — {strategy_name}...')

    # For Undersampling, data is balanced → reset scale_pos_weight to 1 for XGBoost
    if model_name == 'XGBoost':
        rus_model = XGBClassifier(
            scale_pos_weight=1,
            eval_metric='logloss',
            random_state=RANDOM_STATE
        )
    else:
        rus_model = model

    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('rus', RandomUnderSampler(random_state=RANDOM_STATE)),
        ('classifier', rus_model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')
    results.append({
        'Model': model_name,
        'Strategy': strategy_name,
        'Mean F1': scores.mean(),
        'Std F1': scores.std()
    })

print('\nCross-validation complete.')

Training Logistic Regression — No Resampling...
Training Logistic Regression — SMOTE...
Training Logistic Regression — Undersampling...
Training Decision Tree — No Resampling...
Training Decision Tree — SMOTE...
Training Decision Tree — Undersampling...
Training Random Forest — No Resampling...
Training Random Forest — SMOTE...
Training Random Forest — Undersampling...
Training XGBoost — No Resampling...
Training XGBoost — SMOTE...
Training XGBoost — Undersampling...

Cross-validation complete.


## Section 6 — Tableau des Résultats

Les 12 configurations sont comparées dans un seul DataFrame trié par F1 Moyen en ordre décroissant. La colonne Écart-type F1 mesure la stabilité entre les folds — un écart-type faible associé à une moyenne élevée indique une configuration qui généralise de manière fiable.

In [6]:
results_df = pd.DataFrame(results, columns=['Model', 'Strategy', 'Mean F1', 'Std F1'])
results_df['Mean F1'] = results_df['Mean F1'].round(4)
results_df['Std F1']  = results_df['Std F1'].round(4)
results_df = results_df.sort_values('Mean F1', ascending=False).reset_index(drop=True)

display(results_df)

,Model,Strategy,Mean F1,Std F1
0,XGBoost,Undersampling,0.5351,0.0180
1,Logistic Regression,Undersampling,0.5277,0.0286
2,Logistic Regression,No Resampling,0.5249,0.0258
3,XGBoost,No Resampling,0.5239,0.0143
4,Logistic Regression,SMOTE,0.5231,0.0139
5,Random Forest,Undersampling,0.5185,0.0200
6,Random Forest,No Resampling,0.5106,0.0222
7,XGBoost,SMOTE,0.4693,0.0204
8,Decision Tree,Undersampling,0.4559,0.0149
9,Random Forest,SMOTE,0.4514,0.0186


## Section 7 — Justification du Choix du Modèle

D'après les résultats de validation croisée, la meilleure combinaison est **XGBoost + Undersampling** (F1 Moyen = **0.5351**, Écart-type = **0.0180**).

Cette configuration atteint le F1 moyen le plus élevé sur les cinq folds.
Bien que Logistic Regression + Undersampling obtienne un F1 proche (0.5277), XGBoost le dépasse de 0.0074 tout en maintenant un écart-type faible (0.0180), confirmant que les performances sont stables et non dues à un fold favorable.

Undersampling s'impose clairement comme la meilleure stratégie : les 3 premières configurations l'utilisent toutes, devançant systématiquement SMOTE et No Resampling.

**Conclusion :** XGBoost + Undersampling sera retenu pour le tuning en Phase 3 (notebook 05_tuning.ipynb).